# ML v1.3 - сравнение бустингов: CatBoost, LightGBM и XGBoost

Этот notebook сравнивает три сильных бустинга на одном и том же `full_base` наборе признаков: CatBoost, LightGBM и XGBoost. Цель - не менять данные, а проверить, действительно ли CatBoost остается лучшим алгоритмом после `v1.1` и `v1.2`.


In [3]:
from pathlib import Path
import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped:", exc)

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"
RESULTS_DIR = BASE_PATH / "ML" / "1.3" / "results_boosting"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Файл данных:", DATA_FILE)
print("Папка результатов:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Файл данных: /content/drive/MyDrive/ieee-db/client_profile_v1_0_shuffled.csv
Папка результатов: /content/drive/MyDrive/ieee-db/ML/1.3/results_boosting


In [4]:
for package_name, import_name in [
    ("catboost", "catboost"),
    ("lightgbm", "lightgbm"),
    ("xgboost", "xgboost"),
]:
    if importlib.util.find_spec(import_name) is None:
        print(f"{package_name} не найден. Устанавливаю...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
    else:
        print(f"{package_name} уже установлен.")

import time
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)


catboost уже установлен.
lightgbm уже установлен.
xgboost уже установлен.


In [5]:
df = pd.read_csv(DATA_FILE)
print("Размер таблицы:", df.shape)
print("Распределение target:")
print(df["profile_fraud_label"].value_counts(dropna=False))
print("Доля fraud:", df["profile_fraud_label"].mean())
display(df.head())


Размер таблицы: (174280, 95)
Распределение target:
profile_fraud_label
0    167785
1      6495
Name: count, dtype: int64
Доля fraud: 0.03726761533165022


,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_median_proxy,tx_amount_std,tx_count_standart_flow,tx_count_high_risk_flow,tx_sum_stanadart_flow_proxy_proxy,tx_sum_high_risk_flow_proxy,tx_freq_per_day,daily_activity_share,avg_inter_tx_seconds,short_turnover_share,amount_repeat_share,odd_amount_share,weighted_amount_repeat_share,cash_out_ratio_proxy,MCC_risk_share_proxy,high_risk_vs_mean,crypto_pattern_score,low_history_flag,history_support_score,productcd_nunique,addr2_nunique,card4_mode,card6_mode,tx_dt_span_days,top_email_domain,profile_fraud_label,identity_present,num_missing_identity,identity_rows,non_null_id_values,device_type_nunique,id_01_mean,id_01_median,id_01_count,id_02_mean,id_02_median,id_02_count,id_03_mean,id_03_median,id_03_count,id_04_mean,id_04_median,id_04_count,id_05_mean,id_05_median,id_05_count,id_06_mean,id_06_median,id_06_count,id_07_mean,id_07_median,id_07_count,id_08_mean,id_08_median,id_08_count,id_09_mean,id_09_median,id_09_count,id_10_mean,id_10_median,id_10_count,id_11_mean,id_11_median,id_11_count,id_12_mode,id_13_mode,id_14_mode,id_15_mode,id_16_mode,id_17_mode,id_18_mode,id_19_mode,id_20_mode,id_21_mode,id_22_mode,id_23_mode,id_24_mode,id_25_mode,id_26_mode,id_27_mode,id_28_mode,id_29_mode,id_30_mode,id_31_mode,id_32_mode,id_33_mode,id_34_mode,id_35_mode,id_36_mode,id_37_mode,id_38_mode
0,12221_170.0_173,1,47.950,47.950000,47.950,0.000000,1,0,47.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,aol.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5347__14,1,31.191,31.191000,31.191,0.000000,0,1,0.00,31.191,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,1.0,1.0,0.968935,1,0.1,1,0,mastercard,debit,0.000000,gmail.com,0,1,14,1,24,1.0,-10.0,-10.0,1.0,232053.0,232053.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,-2.0,-2.0,1.0,-100.0,-100.0,1.0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,1.0,-6.0,-6.0,1.0,100.0,100.0,1.0,NotFound,52.0,NaN,Found,Found,166.0,13.0,456.0,256.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T
2,7508_420.0_97,3,117.850,39.283333,36.950,7.133645,3,0,117.85,0.000,0.059385,0.058824,2182363.0,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.3,1,1,visa,debit,50.517662,outlook.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17759_126.0_12,1,103.950,103.950000,103.950,0.000000,1,0,103.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16092_143.0_74,3,3426.550,1142.183333,1325.880,259.786317,3,0,3426.55,0.000,3.000000,1.000000,2160.0,1.0,0.666667,1.0,0.133333,1.0,0.0,0.0,0.000000,1,0.3,1,1,mastercard,debit,0.050000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Политика признаков

Используем `full_base`: убираем `client_id`, прямые fraud-утечки и признаки с долей пропусков выше 95%. Это тот набор, который победил в `v1.2` ablation.


In [6]:
TARGET_COL = "profile_fraud_label"
ID_COLS = ["client_id"]
LEAKY_COLS = [c for c in ["tx_count_fraud", "fraud_rate"] if c in df.columns]
HIGH_MISSING_THRESHOLD = 0.95

all_candidate_features = [
    c for c in df.columns
    if c not in [TARGET_COL] + ID_COLS + LEAKY_COLS
]

high_missing_cols = [
    c for c in all_candidate_features
    if df[c].isna().mean() > HIGH_MISSING_THRESHOLD
]

features = [c for c in all_candidate_features if c not in high_missing_cols]
X = df[features].copy()
y = df[TARGET_COL].astype(int).copy()

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Features:", len(features))
print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("High missing excluded:", high_missing_cols)
print("Categorical cols:", categorical_cols)


Features: 82
Numeric: 66
Categorical: 16
High missing excluded: ['id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']
Categorical cols: ['card4_mode', 'card6_mode', 'top_email_domain', 'id_12_mode', 'id_15_mode', 'id_16_mode', 'id_28_mode', 'id_29_mode', 'id_30_mode', 'id_31_mode', 'id_33_mode', 'id_34_mode', 'id_35_mode', 'id_36_mode', 'id_37_mode', 'id_38_mode']


In [7]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1764705882,
    random_state=42,
    stratify=y_train_val,
)

print("Train:", X_train.shape, "fraud rate:", y_train.mean())
print("Val:  ", X_val.shape, "fraud rate:", y_val.mean())
print("Test: ", X_test.shape, "fraud rate:", y_test.mean())


Train: (121996, 82) fraud rate: 0.037271713826682845
Val:   (26142, 82) fraud rate: 0.037258052176574095
Test:  (26142, 82) fraud rate: 0.037258052176574095


In [8]:
def safe_roc_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_pr_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def calc_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": safe_roc_auc(y_true, y_prob),
        "pr_auc": safe_pr_auc(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pred_positive_rate": y_pred.mean(),
        "pred_positive_count": int(y_pred.sum()),
        "support": int(len(y_true)),
        "positive_support": int(y_true.sum()),
    }


def build_threshold_table(y_true, y_prob, thresholds=(0.50, 0.70, 0.75, 0.80, 0.85, 0.90)):
    rows = []
    for threshold in thresholds:
        row = calc_metrics(y_true, y_prob, threshold=threshold)
        row["threshold"] = threshold
        rows.append(row)
    return pd.DataFrame(rows)


def build_topk_table(y_true, y_prob, rates=(0.01, 0.03, 0.05, 0.10, 0.15)):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    order = np.argsort(-y_prob)
    total_positive = y_true.sum()
    rows = []
    for rate in rates:
        k = max(1, int(np.ceil(len(y_true) * rate)))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append({
            "top_rate": rate,
            "top_k": k,
            "precision_at_k": positives_found / k,
            "recall_at_k": positives_found / total_positive if total_positive else 0.0,
            "fraud_found": positives_found,
            "total_fraud": int(total_positive),
        })
    return pd.DataFrame(rows)


## Preprocessing для LightGBM и XGBoost

CatBoost получает категориальные признаки напрямую. LightGBM и XGBoost в этом benchmark получают one-hot представление категориальных признаков, чтобы сравнение было стабильным и не зависело от экспериментальной поддержки categorical в конкретной версии библиотек.


In [9]:
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse=True)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", make_onehot_encoder()),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Fit-transform preprocessing для LightGBM/XGBoost...")
X_train_enc = preprocess.fit_transform(X_train)
X_val_enc = preprocess.transform(X_val)
X_test_enc = preprocess.transform(X_test)

try:
    encoded_feature_names = preprocess.get_feature_names_out()
except Exception:
    encoded_feature_names = np.array([f"f_{i}" for i in range(X_train_enc.shape[1])])

print("Размер encoded train:", X_train_enc.shape)
print("Размер encoded val:", X_val_enc.shape)
print("Размер encoded test:", X_test_enc.shape)


Fit-transform preprocessing для LightGBM/XGBoost...
Размер encoded train: (121996, 345)
Размер encoded val: (26142, 345)
Размер encoded test: (26142, 345)


## Опорный CatBoost

По умолчанию Опорный CatBoost включен, чтобы benchmark был самодостаточным. Если нужно быстро проверить только LightGBM/XGBoost, можно поставить `RUN_CATBOOST_REFERENCE = False`.


In [ ]:
TASK_TYPE = "GPU"
RUN_CATBOOST_REFERENCE = True
RUN_LIGHTGBM = True
RUN_XGBOOST = True

model_results = []
threshold_rows = []
topk_rows = []
feature_importance_rows = []
fitted_models = {}


def record_model_result(model_name, split_name, y_true, y_prob, elapsed_sec, extra=None):
    row = calc_metrics(y_true, y_prob, threshold=0.5)
    row.update({
        "model": model_name,
        "split": split_name,
        "elapsed_sec": elapsed_sec,
    })
    if extra:
        row.update(extra)
    model_results.append(row)

    th = build_threshold_table(y_true, y_prob)
    th["model"] = model_name
    th["split"] = split_name
    threshold_rows.append(th)

    tk = build_topk_table(y_true, y_prob)
    tk["model"] = model_name
    tk["split"] = split_name
    topk_rows.append(tk)


def save_intermediate():
    pd.DataFrame(model_results).to_csv(RESULTS_DIR / "ml_v1_3_model_results.csv", index=False)
    if threshold_rows:
        pd.concat(threshold_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_3_thresholds.csv", index=False)
    if topk_rows:
        pd.concat(topk_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_3_topk.csv", index=False)
    if feature_importance_rows:
        pd.concat(feature_importance_rows, ignore_index=True).to_csv(RESULTS_DIR / "ml_v1_3_feature_importance.csv", index=False)


In [11]:
def prepare_catboost_frame(frame):
    out = frame.copy()
    for c in categorical_cols:
        out[c] = out[c].astype("object").where(out[c].notna(), "MISSING").astype(str)
    return out

if RUN_CATBOOST_REFERENCE:
    print("Обучаю catboost_reference...")
    start = time.time()

    X_train_cb = prepare_catboost_frame(X_train)
    X_val_cb = prepare_catboost_frame(X_val)
    X_test_cb = prepare_catboost_frame(X_test)
    cat_feature_indices = [X_train_cb.columns.get_loc(c) for c in categorical_cols]

    train_pool = Pool(X_train_cb, y_train, cat_features=cat_feature_indices)
    val_pool = Pool(X_val_cb, y_val, cat_features=cat_feature_indices)
    test_pool = Pool(X_test_cb, y_test, cat_features=cat_feature_indices)

    catboost_reference = CatBoostClassifier(
        iterations=3500,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=8,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42,
        task_type=TASK_TYPE,
        thread_count=-1,
        allow_writing_files=False,
        early_stopping_rounds=150,
        verbose=200,
    )
    catboost_reference.fit(train_pool, eval_set=val_pool, use_best_model=True)
    elapsed = time.time() - start
    fitted_models["catboost_reference"] = catboost_reference

    for split_name, pool, y_part in [("val", val_pool, y_val), ("test", test_pool, y_test)]:
        prob = catboost_reference.predict_proba(pool)[:, 1]
        record_model_result(
            "catboost_reference", split_name, y_part, prob, elapsed,
            extra={"best_iteration": catboost_reference.get_best_iteration()},
        )

    fi = pd.DataFrame({"feature": features, "importance": catboost_reference.get_feature_importance()})
    fi["model"] = "catboost_reference"
    feature_importance_rows.append(fi.sort_values("importance", ascending=False))
    save_intermediate()
    print("Опорный CatBoost done. Результаты saved.")
else:
    print("Опорный CatBoost skipped.")


Обучаю catboost_reference...
0:	learn: 0.8176103	test: 0.8075705	best: 0.8075705 (0)	total: 586ms	remaining: 34m 10s


KeyboardInterrupt: 

## LightGBM

Два варианта: default-ish и чуть более регуляризованный. Оба используют один и тот же one-hot preprocessing.


In [ ]:
def fit_lightgbm_model(model_name, params):
    print("\n" + "=" * 100)
    print("Обучаю", model_name)
    start = time.time()

    model = LGBMClassifier(**params)
    model.fit(
        X_train_enc,
        y_train,
        eval_set=[(X_val_enc, y_val)],
        eval_metric="average_precision",
        callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
    )
    elapsed = time.time() - start
    fitted_models[model_name] = model

    for split_name, X_part, y_part in [("val", X_val_enc, y_val), ("test", X_test_enc, y_test)]:
        prob = model.predict_proba(X_part)[:, 1]
        best_iter = getattr(model, "best_iteration_", None)
        record_model_result(model_name, split_name, y_part, prob, elapsed, extra={"best_iteration": best_iter})

    fi = pd.DataFrame({"feature": encoded_feature_names, "importance": model.feature_importances_})
    fi["model"] = model_name
    feature_importance_rows.append(fi.sort_values("importance", ascending=False).head(200))
    save_intermediate()
    print(model_name, "done. Результаты saved.")

if RUN_LIGHTGBM:
    lgb_base_params = dict(
        objective="binary",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=2.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    lgb_tuned_params = dict(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.025,
        num_leaves=31,
        min_child_samples=120,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=4.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    fit_lightgbm_model("lightgbm_default", lgb_base_params)
    fit_lightgbm_model("lightgbm_tuned_small", lgb_tuned_params)
else:
    print("LightGBM skipped.")


## XGBoost

XGBoost использует тот же one-hot preprocessing. Для учета дисбаланса классов задаем `scale_pos_weight = negative / positive`.


In [ ]:
def fit_xgboost_model(model_name, params):
    print("\n" + "=" * 100)
    print("Обучаю", model_name)
    start = time.time()

    model = XGBClassifier(**params)
    model.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=100)
    elapsed = time.time() - start
    fitted_models[model_name] = model

    for split_name, X_part, y_part in [("val", X_val_enc, y_val), ("test", X_test_enc, y_test)]:
        prob = model.predict_proba(X_part)[:, 1]
        best_iter = getattr(model, "best_iteration", None)
        record_model_result(model_name, split_name, y_part, prob, elapsed, extra={"best_iteration": best_iter})

    fi = pd.DataFrame({"feature": encoded_feature_names, "importance": model.feature_importances_})
    fi["model"] = model_name
    feature_importance_rows.append(fi.sort_values("importance", ascending=False).head(200))
    save_intermediate()
    print(model_name, "done. Результаты saved.")

if RUN_XGBOOST:
    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    scale_pos_weight = neg / max(pos, 1)
    print("scale_pos_weight:", scale_pos_weight)

    xgb_base_params = dict(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=2500,
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=5,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=2.0,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=120,
    )

    xgb_tuned_params = dict(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=3000,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=10,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=4.0,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=120,
    )

    fit_xgboost_model("xgboost_default", xgb_base_params)
    fit_xgboost_model("xgboost_tuned_small", xgb_tuned_params)
else:
    print("XGBoost skipped.")


## Итоговое сравнение

Выбираем лучшую модель по validation PR-AUC и подтверждаем на test. Затем смотрим thresholds и top-k для всех моделей.


In [ ]:
results_df = pd.DataFrame(model_results)
thresholds_df = pd.concat(threshold_rows, ignore_index=True) if threshold_rows else pd.DataFrame()
topk_df = pd.concat(topk_rows, ignore_index=True) if topk_rows else pd.DataFrame()
importance_df = pd.concat(feature_importance_rows, ignore_index=True) if feature_importance_rows else pd.DataFrame()

results_df.to_csv(RESULTS_DIR / "ml_v1_3_model_results.csv", index=False)
thresholds_df.to_csv(RESULTS_DIR / "ml_v1_3_thresholds.csv", index=False)
topk_df.to_csv(RESULTS_DIR / "ml_v1_3_topk.csv", index=False)
importance_df.to_csv(RESULTS_DIR / "ml_v1_3_feature_importance.csv", index=False)

print("Saved results to:", RESULTS_DIR)
display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]))

best_model_name = (
    results_df[results_df["split"] == "val"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
print("Лучшая модель по val PR-AUC:", best_model_name)

print("Test rows sorted by PR-AUC:")
display(results_df[results_df["split"] == "test"].sort_values("pr_auc", ascending=False))

print("Top-k test:")
display(topk_df[topk_df["split"] == "test"].sort_values(["top_rate", "precision_at_k"], ascending=[True, False]))

print("Thresholds test:")
display(thresholds_df[thresholds_df["split"] == "test"].sort_values(["model", "threshold"]))


## Как читать v1.3

Если LightGBM или XGBoost заметно обгоняют CatBoost по validation и test PR-AUC, а также дают лучший top-k, следующий шаг - long-run sweep для победителя. Если CatBoost остается лучше или сопоставим, то выбор CatBoost как финального кандидата становится более обоснованным.
